# EDA — Credit Card Fraud Detection (Sparkov)
## CRISP-DM Phase 2 — Data Understanding

**Dataset:** fraudTrain.csv + fraudTest.csv | Period: Jan 2019 - Dec 2020 | 1000 clients, 800 merchants

**Papers:**
- arXiv:2505.00137 (Quantum LSTM, 2025) — Sparkov preprocessing pipeline
- Bahnsen et al. (2019) — Transaction aggregation strategy (74x lift)
- arXiv:2506.02703 (2025) — SMOTE leakage critique
- IEEE 2025 (Hybrid CNN-LSTM+Attention) — Architecture recommandée

In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

Path('notebooks/figures').mkdir(parents=True, exist_ok=True)
DATA_PATH = Path('data/raw/fraud_detection')
print('EDA — Credit Card Fraud Detection (Sparkov)')
print('=' * 50)

In [ ]:
# Cell 2 — Load Data
print('[1] Loading data...')
train = pd.read_csv(DATA_PATH / 'fraudTrain.csv', index_col=0)
test  = pd.read_csv(DATA_PATH / 'fraudTest.csv',  index_col=0)

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'\nColumns: {list(train.columns)}')
train.head(3)

In [ ]:
# Cell 3 — Basic Info
print('[2] Data Types & Missing Values')
print(train.dtypes)
missing = train.isnull().sum()
print(f'\nMissing values: {missing[missing > 0].to_dict() or "None"}')

In [ ]:
# Cell 4 — Class Distribution
train_fraud_rate = train['is_fraud'].mean() * 100
test_fraud_rate  = test['is_fraud'].mean()  * 100
train_counts = train['is_fraud'].value_counts()
test_counts  = test['is_fraud'].value_counts()

print(f'Train — Normal: {train_counts[0]:,} | Fraud: {train_counts[1]:,} | Rate: {train_fraud_rate:.2f}%')
print(f'Test  — Normal: {test_counts[0]:,}  | Fraud: {test_counts[1]:,}  | Rate: {test_fraud_rate:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, counts, rate, title in [
    (axes[0], train_counts, train_fraud_rate, 'Train Set'),
    (axes[1], test_counts,  test_fraud_rate,  'Test Set'),
]:
    ax.pie([counts[0], counts[1]], labels=['Normal', 'Fraud'],
           colors=['#2196F3', '#F44336'], autopct='%1.2f%%', startangle=90)
    ax.set_title(f'{title} — Fraud Rate: {rate:.2f}%')

plt.suptitle('Class Distribution — Extreme Imbalance (SMOTE required)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/figures/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 5 — Amount Analysis
print('[3] Amount Analysis')
print(train.groupby('is_fraud')['amt'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train[train['is_fraud']==0]['amt'].hist(bins=50, alpha=0.7, color='#2196F3', label='Normal', ax=axes[0])
train[train['is_fraud']==1]['amt'].hist(bins=50, alpha=0.7, color='#F44336', label='Fraud',  ax=axes[0])
axes[0].set(xlabel='Amount ($)', ylabel='Count', title='Amount Distribution (log scale)', yscale='log')
axes[0].legend()

train.boxplot(column='amt', by='is_fraud', ax=axes[1])
axes[1].set(title='Amount by Fraud Status', xlabel='is_fraud', ylabel='Amount ($)')

plt.suptitle('Transaction Amount Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/figures/02_amount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6 — Temporal Features (arXiv:2505.00137)
print('[4] Temporal Analysis — features recommended by arXiv:2505.00137')
train['trans_date_trans_time'] = pd.to_datetime(train['trans_date_trans_time'])
train['hour']        = train['trans_date_trans_time'].dt.hour
train['day_of_week'] = train['trans_date_trans_time'].dt.dayofweek
train['month']       = train['trans_date_trans_time'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

for ax, col, labels, title, color in [
    (axes[0], 'hour',        range(24), 'Fraud Rate by Hour of Day',    '#FF9800'),
    (axes[1], 'day_of_week', days,      'Fraud Rate by Day of Week',     '#9C27B0'),
    (axes[2], 'month',       range(1,13),'Fraud Rate by Month',          '#4CAF50'),
]:
    grp = train.groupby(col)['is_fraud'].agg(['sum','count'])
    grp['rate'] = grp['sum'] / grp['count'] * 100
    ax.bar(grp.index if col != 'day_of_week' else days, grp['rate'], color=color, alpha=0.8)
    ax.axhline(train_fraud_rate, color='red', linestyle='--', label=f'Overall: {train_fraud_rate:.2f}%')
    ax.set(xlabel=col, ylabel='Fraud Rate (%)', title=title)
    ax.legend(fontsize=8)

plt.suptitle('Temporal Patterns — Fraud Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/figures/03_temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7 — Category Analysis
print('[5] Fraud Rate by Category')
cat = train.groupby('category')['is_fraud'].agg(['sum','count'])
cat['rate'] = cat['sum'] / cat['count'] * 100
cat = cat.sort_values('rate')
print(cat[['sum','count','rate']].to_string())

colors = ['#F44336' if r > 1.0 else '#FF9800' if r > 0.5 else '#4CAF50' for r in cat['rate']]
fig, ax = plt.subplots(figsize=(12, 6))
cat['rate'].plot(kind='barh', ax=ax, color=colors, alpha=0.8)
ax.axvline(train_fraud_rate, color='red', linestyle='--', label=f'Overall: {train_fraud_rate:.2f}%')
ax.set(xlabel='Fraud Rate (%)', title='Fraud Rate by Transaction Category')
ax.legend()
plt.tight_layout()
plt.savefig('notebooks/figures/04_category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8 — Geospatial Analysis (arXiv:2505.00137 — haversine formula)
print('[6] Geospatial Distance Analysis')

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

sample = train.sample(50000, random_state=42)
sample['distance'] = haversine(sample['lat'], sample['long'], sample['merch_lat'], sample['merch_long'])
print(sample.groupby('is_fraud')['distance'].describe())

fig, ax = plt.subplots(figsize=(10, 5))
sample[sample['is_fraud']==0]['distance'].hist(bins=50, alpha=0.7, color='#2196F3', label='Normal', ax=ax)
sample[sample['is_fraud']==1]['distance'].hist(bins=50, alpha=0.7, color='#F44336', label='Fraud',  ax=ax)
ax.set(xlabel='Distance (km)', ylabel='Count',
       title='Distance Customer ↔ Merchant\n(Sparkov: merchants placed <150km by design)')
ax.legend()
plt.tight_layout()
plt.savefig('notebooks/figures/05_geospatial_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9 — Transaction Velocity (Bahnsen et al. 2019 — 74x lift)
print('[7] Transaction Velocity — Bahnsen et al. 2019 (74x lift feature)')
sample_sorted = sample.sort_values(['cc_num', 'unix_time'])
sample_sorted['tx_count_cumul'] = sample_sorted.groupby('cc_num').cumcount()
print(sample_sorted.groupby('is_fraud')['tx_count_cumul'].describe())

fig, ax = plt.subplots(figsize=(10, 5))
sample_sorted.boxplot(column='tx_count_cumul', by='is_fraud', ax=ax)
ax.set(title='Cumulative Transaction Count by Fraud Status',
       xlabel='is_fraud (0=Normal, 1=Fraud)', ylabel='Transaction Count')
plt.suptitle('')
plt.tight_layout()
plt.savefig('notebooks/figures/06_velocity_feature.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10 — Correlation Heatmap
print('[8] Feature Correlation Matrix')
num_cols = ['amt','lat','long','city_pop','unix_time','merch_lat','merch_long',
            'hour','day_of_week','month','is_fraud']
corr = train[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/figures/07_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11 — EDA Summary
print('=' * 60)
print('EDA SUMMARY — CRISP-DM Phase 2 Complete')
print('=' * 60)
print(f"""
Dataset:
  Train: {train.shape[0]:,} transactions | Test: {test.shape[0]:,} transactions
  Fraud rate train: {train_fraud_rate:.2f}% | test: {test_fraud_rate:.2f}%
  Period: Jan 2019 — Dec 2020 | 1000 clients | 800 marchands

Key Findings (→ decisions Phase 3):
  1. EXTREME IMBALANCE 0.58% → SMOTE sur train uniquement (arXiv:2506.02703)
  2. MIDNIGHT FRAUD → feature 'hour' critique pour CNN+LSTM
  3. TOP CATEGORIES FRAUDE → shopping_net, misc_net, grocery_pos
  4. DISTANCE limitée → Sparkov <150km (documenter dans limites du rapport)
  5. VELOCITY 74x lift → tx_count_24h feature (Bahnsen et al. 2019)

Features retenus pour Phase 3:
  Temporelles : hour, day_of_week, month
  Géospatiales : distance haversine
  Montant      : log(amt+1)
  Catégorie    : one-hot (14 catégories)
  Client       : age, gender
  Vélocité     : tx_count_24h (Bahnsen 2019)

Next Step: CRISP-DM Phase 3 — Data Preparation
""")